# B-030 diagnostics: why is the harmful endpoint not below the unharmful one? (CPU only)

**Notebook settings:** Accelerator **None (CPU)** · Internet **on** · Secrets: **HF_TOKEN**. Run as **Save Version → Save & Run All**.

The notebook restores the M4 caches from the private store (`manifests`, `sentences`, `embed`, `embeddings/<slug>`; ≈ 10 GB in total). For each config it then runs `python -m rbbd.cli diagnose-deltab --config <cfg>` (`rbbd.analysis.diagnose`).

For each spectrum it reports:
- method agreement (RR, SEAT, Procrustes);
- the α trend;
- the harmful − unharmful contrast, with a 95% CI from a bootstrap over templates;
- an S⁺/S⁻ decomposition;
- a group-free control (the Alpaca pool) that separates generic drift from group bias;
- geometry (P/N centroid cosine, anisotropy);
- ΔB per group.

Output is aggregates only. No sentence text is printed.


In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "main"  # a branch, a tag, or a FULL 40-character commit SHA (short SHAs fail)
WORK = "/kaggle/working/rbbd"
CONFIGS = ["tier2_qwen2.5-0.5b", "tier2_llama3.2-1b", "tier1_llama3.1-8b", "tier1_mistral-7b", "tier1_gemma3-4b"]
import time
SESSION_T0 = time.time()


In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
!pip install -q -e ".[train,dev]" 2>&1 | tail -5
!pip freeze > /kaggle/working/pip_freeze_b030.txt
# Preflight (B-012): fails here, not inside a stage, if torch and torchvision do not match.
!python -c "import torch, torchvision, transformers; from transformers import Qwen2ForCausalLM, LlamaForCausalLM, Gemma3ForConditionalGeneration; print('preflight ok', torch.__version__, torchvision.__version__, transformers.__version__)"

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# Restore the M4 caches, then diagnose each config. Aggregates only.
import json, shutil
from pathlib import Path
from rbbd import config as _config
ROOT = Path("/kaggle/working/artifacts")
for path in ["manifests", "sentences", "embed"]:
    !python -m rbbd.cli restore --path {path}
REPORTS = {}
for NAME in CONFIGS:
    cfg = _config.load(f"configs/{NAME}.yaml")
    SLUG, RUN = cfg["models"][0]["slug"], cfg["run_name"]
    print(f"===== {NAME} =====")
    !python -m rbbd.cli restore --path embeddings/{SLUG}
    !python -m rbbd.cli diagnose-deltab --config configs/{NAME}.yaml > /kaggle/working/b030_{NAME}.json
    res = Path(WORK) / "results" / RUN / "b030_diagnostics.json"
    if not res.exists():
        print("FAILED:", NAME); continue
    REPORTS[NAME] = json.loads(res.read_text())
    shutil.copytree(res.parent, ROOT / "results" / RUN, dirs_exist_ok=True)
    !python -m rbbd.cli sync --path results/{RUN}


In [ ]:
# Compact summary per spectrum (paste back), then the full reports.
def r3(x): return None if x is None else round(x, 4)
for name, rep in REPORTS.items():
    for spec, d in rep["spectra"].items():
        m = d["methods"]; c = d.get("contrast", {})
        print(json.dumps({
            "spectrum": spec,
            "a100": {k: r3(m["a100"][k]) for k in ("rr", "seat", "procrustes")},
            "a000": {k: r3(m["a000"][k]) for k in ("rr", "seat", "procrustes")},
            "alpha_trend": {k: r3(v) for k, v in d["alpha_trend"].items()},
            "contrast": {k: {"mean": r3(v["mean_h_minus_u"]), "ci95": [r3(x) for x in v["ci95"]],
                             "groups_h_below_u": v["groups_h_below_u"]} for k, v in c.items()},
            "decomposition": {k: {kk: r3(vv) for kk, vv in v.items()} for k, v in d["decomposition"].items()},
            "control": {k: {kk: r3(vv) for kk, vv in v.items()} for k, v in d["control"].items()},
            "geometry": {k: {kk: r3(vv) for kk, vv in v.items()} for k, v in d["geometry"].items()},
        }))
print("session hours:", round((time.time() - SESSION_T0) / 3600, 2))
!df -h /kaggle/working | tail -1
for name, rep in REPORTS.items():
    print(f"----- {name} (full)"); print(json.dumps(rep, indent=1))
